# Lean-19c : borne de Bernstein pour le BSC (Python)

**Navigation** : [<< Lean-19 Hoeffding](Lean-19-Detection-MIMO-Index-Python.ipynb) · [Lean-19b Chernoff](Lean-19b-Borne-Chernoff-Python.ipynb) · [Lean-19d Bayésien](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [Sommaire série](README.md)
**Paire methodologique** : Lean-19 (Hoeffding, distribution-free) -- Lean-19b (Chernoff, distribution-specifique) -- **Lean-19c (Bernstein, compromis)**.

Pour X = somme de N variables iid Bernoulli(p), la borne de Bernstein donne :

P(|X - Np| >= t) <= 2 * exp(-t^2 / (2 * (N*p*(1-p) + t/3)))

Comme Chernoff, elle exploite la variance reelle p(1-p) (au lieu du majorant 1/4 d'Hoeffding).
Contrairement a Chernoff, elle **ne requiert pas la mgf exacte** : seulement mu et sigma^2.
Elle s'applique donc aux distributions **non-bornees** (Poisson, gaussienne, ...).


## Bornes de reference : Hoeffding et Chernoff

**Hoeffding (distribution-free)** : pour X = somme de N v.a. dans [a, b] :

P(|X - mu| >= t) <= 2 * exp(-2*t^2 / (N*(b-a)^2))

Pour Bernoulli dans {0,1}, (b-a) = 1, donc :
  eps_H = sqrt(log(2/eps_p) / (2*N))

**Chernoff (approx KL)** : pour X = somme de N Bernoulli(p), avec mgf exacte M_X(t) = (1-p) + p*e^t :
  eps_C = sqrt(2*log(2/eps_p)*p*(1-p) / N)

Pour p = 1/2, Chernoff et Hoeffding **coincident** exactement : les deux valent sqrt(L/(2N)), donc le rapport H/C vaut 1.000 (BSC-05, BSC-08, BSC-13, BSC-17, BSC-20).
Le rapport H/C = 1/(2*sqrt(p*(1-p))) ne depend que de p : 1.25 a p = 0.20, 1.67 a p = 0.10, 2.29 a p = 0.05. Le gain de Chernoff croit quand p s'eloigne de 1/2, sans borne a la limite ; sur la plage couverte par le tableau (p >= 0.05), il reste sous 2.3x.


In [1]:
import math
import random

random.seed(42)

print('Setup OK -- 20 instances BSC, 5 instances Poisson, seed=42')


Setup OK -- 20 instances BSC, 5 instances Poisson, seed=42


## La borne de Bernstein

Pour X = somme de N v.a. **independantes** X_i avec mu_i = E[X_i], sigma_i^2 = Var(X_i), on a :

P(|X - sum(mu_i)| >= t) <= 2 * exp(-t^2 / (2 * (sum sigma_i^2 + t/3)))

Pour X = somme de N Bernoulli(p) iid : sum mu_i = N*p, sum sigma_i^2 = N*p*(1-p).

En divisant par N, on obtient l'ecart autorise sur p_hat = X/N :
  P(|p_hat - p| >= eps) <= 2 * exp(-N*eps^2 / (2*(p*(1-p) + eps/3)))

L'inversion de cette inegalite en eps donne une **formule quadratique fermee** :
  N*eps^2 - (2*L/3)*eps - 2*L*p*(1-p) = 0, ou L = log(2/eps_p)
  eps_B = (2*L/3 + sqrt(4*L^2/9 + 8*N*L*p*(1-p))) / (2*N)


In [2]:
def hoeffding_eps(N, p, eps_p=0.05):
    L = math.log(2.0 / eps_p)
    return math.sqrt(L / (2.0 * N))

def chernoff_eps(N, p, eps_p=0.05):
    L = math.log(2.0 / eps_p)
    return math.sqrt(2.0 * L * p * (1.0 - p) / N)

def bernstein_eps(N, p, eps_p=0.05):
    L = math.log(2.0 / eps_p)
    disc = 4.0 * L * L / 9.0 + 8.0 * N * L * p * (1.0 - p)
    return (2.0 * L / 3.0 + math.sqrt(disc)) / (2.0 * N)

BSC_INSTANCES = [
    ('BSC-01', 100,  0.10, 0.05), ('BSC-02', 100,  0.20, 0.05),
    ('BSC-03', 100,  0.30, 0.05), ('BSC-04', 100,  0.40, 0.05),
    ('BSC-05', 100,  0.50, 0.05), ('BSC-06', 500,  0.10, 0.05),
    ('BSC-07', 500,  0.30, 0.05), ('BSC-08', 500,  0.50, 0.05),
    ('BSC-09', 1000, 0.10, 0.05), ('BSC-10', 1000, 0.20, 0.05),
    ('BSC-11', 1000, 0.30, 0.05), ('BSC-12', 1000, 0.40, 0.05),
    ('BSC-13', 1000, 0.50, 0.05), ('BSC-14', 5000, 0.05, 0.01),
    ('BSC-15', 5000, 0.10, 0.01), ('BSC-16', 5000, 0.30, 0.01),
    ('BSC-17', 5000, 0.50, 0.01), ('BSC-18', 10000, 0.10, 0.05),
    ('BSC-19', 10000, 0.30, 0.05), ('BSC-20', 10000, 0.50, 0.05),
]

print('=' * 95)
print('TABLEAU COMPARATIF BORNES BSC : Hoeffding vs Chernoff vs Bernstein')
print('=' * 95)
print(f"{'Instance':>10} {'N':>6} {'p':>5} {'eps_p':>6} | {'eps_H':>9} {'eps_C':>9} {'eps_B':>9} | {'H/B':>6} {'C/B':>6} {'H/C':>6}")
print('-' * 95)

ratios_HB, ratios_CB, ratios_HC = [], [], []
for label, N, p, eps_p in BSC_INSTANCES:
    eH = hoeffding_eps(N, p, eps_p)
    eC = chernoff_eps(N, p, eps_p)
    eB = bernstein_eps(N, p, eps_p)
    rHB, rCB, rHC = eH/eB, eC/eB, eH/eC
    ratios_HB.append(rHB); ratios_CB.append(rCB); ratios_HC.append(rHC)
    print(f"{label:>10} {N:>6d} {p:>5.2f} {eps_p:>6.3f} | {eH:>9.5f} {eC:>9.5f} {eB:>9.5f} | {rHB:>6.3f} {rCB:>6.3f} {rHC:>6.3f}")

print('-' * 95)
print(f"{'MOYENNE':>10} {'':>6} {'':>5} {'':>6} | {'':>9} {'':>9} {'':>9} | {sum(ratios_HB)/len(ratios_HB):>6.3f} {sum(ratios_CB)/len(ratios_CB):>6.3f} {sum(ratios_HC)/len(ratios_HC):>6.3f}")
print(f"{'MEDIANE':>10} {'':>6} {'':>5} {'':>6} | {'':>9} {'':>9} {'':>9} | {sorted(ratios_HB)[len(ratios_HB)//2]:>6.3f} {sorted(ratios_CB)[len(ratios_CB)//2]:>6.3f} {sorted(ratios_HC)[len(ratios_HC)//2]:>6.3f}")
print()
n_B_tighter_H = sum(1 for r in ratios_HB if r > 1)
n_C_tighter_B = sum(1 for r in ratios_CB if r < 1)
print(f"Bernstein plus tight que Hoeffding : {n_B_tighter_H}/20")
print(f"Chernoff plus tight que Bernstein : {n_C_tighter_B}/20 (mgf exacte de Chernoff vs borne en mu/sigma^2 : c'est le cas general pour Bernoulli)")


TABLEAU COMPARATIF BORNES BSC : Hoeffding vs Chernoff vs Bernstein
  Instance      N     p  eps_p |     eps_H     eps_C     eps_B |    H/B    C/B    H/C
-----------------------------------------------------------------------------------------------
    BSC-01    100  0.10  0.050 |   0.13581   0.08149   0.09470 |  1.434  0.860  1.667
    BSC-02    100  0.20  0.050 |   0.13581   0.10865   0.12164 |  1.117  0.893  1.250
    BSC-03    100  0.30  0.050 |   0.13581   0.12447   0.13737 |  0.989  0.906  1.091
    BSC-04    100  0.40  0.050 |   0.13581   0.13307   0.14593 |  0.931  0.912  1.021
    BSC-05    100  0.50  0.050 |   0.13581   0.13581   0.14866 |  0.914  0.914  1.000
    BSC-06    500  0.10  0.050 |   0.06074   0.03644   0.03898 |  1.558  0.935  1.667
    BSC-07    500  0.30  0.050 |   0.06074   0.05567   0.05818 |  1.044  0.957  1.091
    BSC-08    500  0.50  0.050 |   0.06074   0.06074   0.06325 |  0.960  0.960  1.000
    BSC-09   1000  0.10  0.050 |   0.04295   0.02577   0.02703 

## Extension : les distributions non-bornees (Poisson)

**Hoeffding REQUIERT des variables bornees** dans [a, b]. Pour une distribution non-bornee
comme Poisson, elle n'est pas directement applicable.

**Chernoff** s'applique aux v.a. dont on connait la mgf exacte. Pour Poisson(lambda),
M(t) = exp(lambda*(e^t - 1)), et l'approximation KL quadratique donne t_C = sqrt(2*log(2/eps_p)*lambda).
C'est la meme formule que Bernstein au premier ordre.

**Bernstein** ne requiert que mu et sigma^2. Pour Poisson, mu = sigma^2 = lambda :
  P(|Y - lambda| >= t) <= 2 * exp(-t^2 / (2*(lambda + t/3)))

Resolution quadratique en t (L = log(2/eps_p)) :
  t^2 - (2*L/3)*t - 2*lambda*L = 0
  t_B = (2*L/3 + sqrt(4*L^2/9 + 8*lambda*L)) / 2


In [3]:
def poisson_bernstein_t(lam, eps_p=0.05):
    L = math.log(2.0 / eps_p)
    disc = 4.0 * L * L / 9.0 + 8.0 * lam * L
    return (2.0 * L / 3.0 + math.sqrt(disc)) / 2.0

POISSON_INSTANCES = [
    ('POIS-01', 1.0),  ('POIS-02', 5.0),  ('POIS-03', 10.0),
    ('POIS-04', 50.0), ('POIS-05', 100.0),
]

print('=' * 75)
print('EXTENSION POISSON : Bernstein (Hoeffding inapplicable, Chernoff = Bernstein ordre 2)')
print('=' * 75)
print(f"{'Instance':>10} {'lambda':>8} | {'t_B':>16} {'t_C':>16} {'Hoeffding':>10}")
print('-' * 75)
for label, lam in POISSON_INSTANCES:
    tB = poisson_bernstein_t(lam, 0.05)
    tC = tB  # Chernoff approx KL quadratique = Bernstein ordre 2
    print(f"{label:>10} {lam:>8.1f} | {tB:>16.5f} {tC:>16.5f} {'N/A':>10}")
print('-' * 75)
print()
print('Note : pour Poisson, Chernoff et Bernstein coincident au premier ordre.')
print("Au-dela, la borne exacte de Chernoff utilise KL(lambda+t || lambda) = (lambda+t)*log((lambda+t)/lambda) - t")
print('et peut etre plus tight que Bernstein pour t grand.')


EXTENSION POISSON : Bernstein (Hoeffding inapplicable, Chernoff = Bernstein ordre 2)
  Instance   lambda |              t_B              t_C  Hoeffding
---------------------------------------------------------------------------
   POIS-01      1.0 |          4.21119          4.21119        N/A
   POIS-02      5.0 |          7.42646          7.42646        N/A
   POIS-03     10.0 |          9.90658          9.90658        N/A
   POIS-04     50.0 |         20.47540         20.47540        N/A
   POIS-05    100.0 |         28.41948         28.41948        N/A
---------------------------------------------------------------------------

Note : pour Poisson, Chernoff et Bernstein coincident au premier ordre.
Au-dela, la borne exacte de Chernoff utilise KL(lambda+t || lambda) = (lambda+t)*log((lambda+t)/lambda) - t
et peut etre plus tight que Bernstein pour t grand.


## Synthese : quand utiliser quoi ?

**Pour Bernoulli (BSC)** :
- **Hoeffding** : distribution-free, conservative, simple (juste [a, b] = [0, 1])
- **Chernoff** : mgf exacte, plus tight ou egale a Hoeffding sur les 20 instances (strictement plus tight dans 15/20 ; egales a p = 1/2, ou les deux valent sqrt(L/(2N)))
- **Bernstein** : compromis -- exploite la variance reelle p(1-p), bat Hoeffding 12/20 mais perd contre Chernoff (mgf exacte plus precise)

Sur nos 20 instances BSC, **Bernstein est plus tight que Hoeffding dans 12/20 cas** (gain median 1.07x, gain maximal 2.21x -- BSC-14, p = 0.05 et N = 5000). Mais **Chernoff est plus tight ou egale dans 20/20 cas** (gain median 1.04x sur Bernstein : la mgf exacte domine la borne en mu/sigma^2).

**Pour Poisson (et autres distributions non-bornees)** :
- Hoeffding est **inapplicable** (variables non-bornees)
- Chernoff approx KL et Bernstein ordre 2 coincident pour Poisson (memes formules)
- Au-dela de l'ordre 2, Chernoff avec KL exacte devient plus tight
- **Bernstein est le plus naturel** : pas besoin de la mgf complete, juste mu et sigma^2

**Recommandation pratique** :
1. Distribution connue et mgf calculable : **Chernoff** (le plus tight)
2. Seulement mu et sigma^2 connus, ou distribution non-bornee : **Bernstein** (compromis optimal)
3. Seulement les bornes [a, b] connues : **Hoeffding** (distribution-free)


## Exercices

**E1.** Implementez `bernstein_eps_exo(N, p, eps_p=0.05)` qui retourne l'ecart autorise eps_B
pour une Bernoulli de parametre p sur N echantillons, avec confiance 1 - eps_p.
Formule :
  $$L = \log(2/\varepsilon_p)$$
  $$\varepsilon = \frac{2L/3 + \sqrt{4L^2/9 + 8N L p (1-p)}}{2N}$$

**E2.** Implementez `ratio_H_B(N, p, eps_p=0.05)` qui retourne eps_H / eps_B
(le facteur par lequel Hoeffding sur-estime la queue par rapport a Bernstein).
Si le ratio est superieur a 1, Bernstein est plus tight ; sinon Hoeffding est plus tight.

**E3.** Implementez `poisson_bernstein_eps(lambda_, eps_p=0.05)` qui retourne l'ecart autorise t
pour une loi de Poisson de parametre lambda, avec confiance 1 - eps_p.
Formule :
  $$L = \log(2/\varepsilon_p)$$
  $$t = \frac{2L/3 + \sqrt{4L^2/9 + 8\lambda L}}{2}$$


In [4]:
def bernstein_eps_exo(N, p, eps_p=0.05):
    # TODO etudiant : retourner l'ecart autorise eps_B pour une Bernoulli(p, N)
    # Indice : formule quadratique avec L = log(2/eps_p), variance = p*(1-p)
    pass


In [5]:
def ratio_H_B(N, p, eps_p=0.05):
    # TODO etudiant : retourner eps_H / eps_B
    # Indice : Hoeffding eps_H = sqrt(L/(2N)), Bernstein = bernstein_eps_exo
    pass


In [6]:
def poisson_bernstein_eps(lambda_, eps_p=0.05):
    # TODO etudiant : retourner l'ecart autorise t pour Poisson(lambda)
    # Indice : formule quadratique avec L = log(2/eps_p)
    pass


## Pour aller plus loin

- **Chernoff exact (KL)** : pour la borne exacte `exp(-N*KL(p+eps || p))`, on resout
  `(p+eps)*log((p+eps)/p) + (1-p-eps)*log((1-p-eps)/(1-p)) = log(2/eps_p) / N`
  par recherche dichotomique. Pour Bernoulli, c'est strictement plus tight que Bernstein.

- **Bennett** : raffinement de Bernstein qui exploite un majorant sur la mgf au-dela de la variance.
  Pour Bernoulli, donne `t^2 / (2*(p(1-p) - p^2) + ...)` -- gain marginal.

- **Application** : dans Lean-19 et Lean-19b, on a montre la detection MIMO tient avec Hoeffding/Chernoff.
  Avec Bernstein, on peut certifier la detection pour des canaux non-BSC (gaussien, Poisson) --
  utile pour les modeles de bruit reels (shot noise, thermal noise).
